# BRMARL v2 — синхронное пакетное обучение

4 CPU-процесса обслуживают 12 рынков; один родитель владеет GPU и пакетирует neural inference.
Индивидуальные политики десяти курьеров реплицируются между рынками. Все neural weights, CFR и OSR
фиксированы до завершения последней смены пакета. Затем — одно полное actor-обновление на агента,
обучение контекста и обновление библиотек. Online-адаптация вынесена в отдельный тест.

По умолчанию **original**: 385 полигонов, 20160 столбцов заказов и существующий направленный seconds memmap.
Сохранённые fixture outputs — только инженерный прогон. Для Windows/Jupyter workers находятся в .py,
используется spawn; notebook не содержит функций, которые нужно сериализовать в дочерний процесс.

In [ ]:
!pip install -r requirements.txt

In [1]:
from pathlib import Path
import os, sys, json, time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, IFrame
ROOT=Path.cwd()
assert (ROOT/'brmarl').is_dir(), 'Запустите notebook из распакованного проекта'
sys.path.insert(0,str(ROOT))
from brmarl.config import MarketConfig, AgentConfig, ParallelConfig
from brmarl.data import (load_assets, make_fixture_assets, prepare_route_cache,
                         create_search_index, prewarm_centroid_pool)
from brmarl.experiment import train, test, load_checkpoint
from brmarl.evaluation import refusal_metrics
MODE=os.environ.get('BRMARL_MODE','original')
assert MODE in ('original','fixture')
FIXTURE=MODE=='fixture'
DATA_ROOT=Path(os.environ.get('BRMARL_DATA_ROOT',ROOT))
DATA={'properties':'global_properties.pkl','map':'AUG_global_map_osrm_seconds.pkl',
      'results':'epos_cache_fd.pkl','couriers':'courier_training_checkpoint_fd_seconds.pkl',
      'preferences':'glob_pref_map.pkl'}
if FIXTURE:
    assets=make_fixture_assets(ROOT/'data/fixture.json')
else:
    normalized=DATA_ROOT/'brmarl_assets.json'
    assets=load_assets(normalized,config=MarketConfig()) if normalized.exists() else load_assets(
        DATA_ROOT/DATA['map'],DATA_ROOT/DATA['preferences'],config=MarketConfig())
print('MODE:',MODE,'source:',assets['source'])
print('polygons:',len(assets['polygons']),'order columns:',sum(len(p['points']) for p in assets['polygons']))


MODE: original source: original_MPID_AUG
polygons: 385 order columns: 13549


In [2]:
market_cfg=MarketConfig(seed=42,B=43200,n_couriers=10,target_new_orders=300,cap_new_orders=300,
    expected_original_polygons=None if FIXTURE else 385,
    expected_order_points=None if FIXTURE else 20160)
agent_cfg=AgentConfig(seed=42,model_adv_weight=0.)
parallel_cfg=ParallelConfig(workers=4,markets_per_batch=12,device='auto',progress=True)
N_TRAIN_BATCHES=int(os.environ.get('BRMARL_TRAIN_BATCHES','20'))
TRAIN_DIR=ROOT/'outputs'/MODE/'train'
CACHE_ROOT=(ROOT/'outputs'/MODE/'routes') if FIXTURE else DATA_ROOT/'brmarl_route_cache'
print('Batches:',N_TRAIN_BATCHES,'total markets:',N_TRAIN_BATCHES*parallel_cfg.markets_per_batch)


Batches: 20 total markets: 240


## Кэш маршрутов до запуска работников

`point_ids` из исходного `full map` связывают координаты с абсолютными ID memmap. Для старого normalized JSON
переэкспортируйте assets или передайте coordinate_to_id. Штрафов между полигонами нет.

Создаются два отдельных файла одинаковой формы [centroid, order]:
forward[c,o] = centroid[c]→order[o], reverse[c,o] = order[o]→centroid[c]. Значения не транспонируются.
Работники открывают их read-only; дополнения попадают в общий SQLite WAL и сливаются родителем после барьера.
Неизвестные пары идут в существующий cache/OSRM. Продолжительность исполнения — два фактических времени маршрута.

In [3]:
# Выполняется до создания CPU-процессов. Можно передать уже открытые переменные
# distance_array (memmap), idx_dict и coordinate_to_id из исходного notebook.
prep_start=time.perf_counter()
legacy=None;index=None
if not FIXTURE:
    if 'distance_array' in globals() and 'idx_dict' in globals():
        legacy=distance_array;index=idx_dict
    elif not (CACHE_ROOT/'route_spec.json').exists():
        distance_path=Path(os.environ.get('BRMARL_DISTANCE_FILE',DATA_ROOT/'distances.dat'))
        if not distance_path.exists():raise FileNotFoundError('Нужен существующий distances.dat или готовый route_spec.json')
        assert distance_path.stat().st_size%12==0, 'Ожидается raw int32 [from,to,seconds]'
        legacy=np.memmap(distance_path,dtype=np.int32,mode='r',shape=(distance_path.stat().st_size//12,3))
        index=create_search_index(legacy)
ROUTE_SPEC=prepare_route_cache(assets,CACHE_ROOT,legacy,index,
    coordinate_to_id=globals().get('coordinate_to_id'),fixture=FIXTURE,config=market_cfg)
route_info=json.loads(Path(ROUTE_SPEC).read_text())
print('Directed centroid maps:',route_info['shape_centroids'],'each; unknown=NaN')
print('Legacy array:',route_info.get('distance_path'))
print('Cache preparation seconds:',round(time.perf_counter()-prep_start,3))


Directed centroid maps: [385, 13549] each; unknown=NaN
Legacy array: C:\Users\Administrator\distances.dat
Cache preparation seconds: 7.727


In [4]:
PREWARM_ALL=False
CENTROID_POOL=None
ORDER_POOL=None
if PREWARM_ALL or ORDER_POOL is not None:
    display(prewarm_centroid_pool(ROUTE_SPEC,CENTROID_POOL,ORDER_POOL,reverse=True,progress=True))


## Пакеты смен

12 progress bars принадлежат только родителю. В терминале позиции фиксированы, в Jupyter используются widgets.
Модельные часы доходят до B, затем отдельно показываются завершение обязательств, ожидание барьера и learning.
Слоты переиспользуются в следующем пакете. Для файлового stdout bars отключаются.

Теневые цены busy-курьера исходят из последней завершённой/послеперерывной точки. Это гипотеза
о пропуске текущего обязательства, а не его фактическое положение и не достоверная альтернативная смена.
Такие метки не обучают допустимое участие. Реальный победитель обязан исполнить заказ.

In [ ]:
agents,training_summary=train(assets,market_cfg,agent_cfg,n_batches=N_TRAIN_BATCHES,
    output=TRAIN_DIR,fixture=FIXTURE,parallel_cfg=parallel_cfg,route_spec=ROUTE_SPEC)
display(training_summary.groupby('batch')[['completed','net_profit','break_cost','return_cost']].sum())


Training: 4 CPU workers / 12 markets / device=cuda


Market 0000:   0%|          | 0/12.0 [00:00<?, ?model h/s]

Market 0001:   0%|          | 0/12.0 [00:00<?, ?model h/s]

Market 0002:   0%|          | 0/12.0 [00:00<?, ?model h/s]

Market 0003:   0%|          | 0/12.0 [00:00<?, ?model h/s]

Market 0004:   0%|          | 0/12.0 [00:00<?, ?model h/s]

... (4 outputs left out)

Market 0009:   0%|          | 0/12.0 [00:00<?, ?model h/s]

Market 0010:   0%|          | 0/12.0 [00:00<?, ?model h/s]

Market 0011:   0%|          | 0/12.0 [00:00<?, ?model h/s]

Batch 0: all 12 markets finished; learning starts
Batch 0: rollout/barrier=31588.4s, learning=14.8s, mean neural batch=1.0


In [6]:
timings=json.loads((TRAIN_DIR/'batch_timings.json').read_text())
display(pd.DataFrame([{k:v for k,v in row.items() if k!='agent_updates'} for row in timings]))
display(pd.DataFrame([dict(batch=row['batch'],courier=i,**update)
    for row in timings for i,update in enumerate(row['agent_updates'])]))
print('Checkpoint:',TRAIN_DIR/'checkpoint.pt')
print('Наблюдайте mean_agent_batch: microbatch size зависит от готовности рынков, не обязан равняться 12.')


Checkpoint: /workspace/scratch/fc90c82b7d15/brmarl_v2/outputs/fixture/train/checkpoint.pt
Наблюдайте mean_agent_batch: microbatch size зависит от готовности рынков, не обязан равняться 12.


,rollout_barrier_wall_s,device,rpc_microbatches,observations,mean_agent_batch,max_agent_batch,inference_wall_s,merge_wall_s,learning_wall_s,cache_merge_wall_s,batch,markets,last_market,policy_version,checkpoint_wall_s
0,134.787595,cpu,11198,37400,1.19941,5,36.343139,5.661773,1.517063,0.047017,0,12,11,1,2.642863


,batch,courier,samples,loss,max_behavior_logprob_error
0,0,0,1459,0.904094,0.000175
1,0,1,1477,0.878584,0.000012
2,0,2,1460,0.926093,0.000005
3,0,3,1475,0.840473,0.000009
4,0,4,1450,0.862097,0.000065
5,0,5,1468,0.917629,0.000061
6,0,6,1476,0.878312,0.000006
7,0,7,1493,0.878156,0.000006
8,0,8,1453,0.861592,0.000004
9,0,9,1463,0.912064,0.000213


## Продолжение

`train(..., resume=TRAIN_DIR/'checkpoint.pt', n_batches=...)` добавляет пакеты к существующему checkpoint.
Checkpoint сохраняется после каждого полного пакета; v1 checkpoint не загружается молча.
Число рынков — бюджет данных, число пакетов — бюджет обновлений. Качество определяется в отдельных тестах,
по seed, калибровке порога, AL и абляциям; двадцать пакетов не являются критерием сходимости.

Все журналы сохранены по batch/shift. В новых логах positions — зафиксированные точки отправления,
а карты показывают endpoint schematics, а не физическую дорожную геометрию.